# Dostrajanie modelu LLM do klasyfikacji zwrotek polskich utworów rapowych na ich wykonawców.


## Instalacja, import bibliotek i patchowanie


In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import TaskType
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

os.environ["UNSLOTH_DISABLE_FAST_GENERATION"] = "1"

# Unsloth przed transformers (optymalizacje + patch Gemma4)
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

from transformers import (
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from transformers.configuration_utils import PretrainedConfig
from transformers.modeling_layers import GenericForSequenceClassification
from transformers.models.auto.modeling_auto import (
    MODEL_FOR_SEQUENCE_CLASSIFICATION_MAPPING,
)
from transformers.models.gemma4.configuration_gemma4 import Gemma4Config
from transformers.models.gemma4.modeling_gemma4 import Gemma4PreTrainedModel

# Workaround: patch Unsloth ukrywa num_kv_shared_layers==0 i psuje AutoConfig.__repr__
_orig_to_diff_dict = PretrainedConfig.to_diff_dict

def _safe_to_diff_dict(self):
    if "Gemma4" in self.__class__.__name__:
        return self.to_dict()
    return _orig_to_diff_dict(self)


PretrainedConfig.to_diff_dict = _safe_to_diff_dict

# Transformers 5.10 nie ma jeszcze gotowej klasy dla Gemma 4 - rejestrujemy ją ręcznie.
class Gemma4ForSequenceClassification(GenericForSequenceClassification, Gemma4PreTrainedModel):
    pass


MODEL_FOR_SEQUENCE_CLASSIFICATION_MAPPING.register(
    Gemma4Config, Gemma4ForSequenceClassification, exist_ok=True
)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/opt/conda/lib/python3.12/site-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Wczytanie datasetu


In [2]:
DATASET_DIR = "dataset"

train_df = pd.read_json(os.path.join(DATASET_DIR, "train.json"))
test_df = pd.read_json(os.path.join(DATASET_DIR, "test.json"))

print(f"Zbiór treningowy: {len(train_df)} zwrotek")
print(f"Zbiór testowy:    {len(test_df)} zwrotek")
print(f"Liczba unikalnych etykiet ze zbioru treningowego: {train_df['label'].nunique()}")
train_df.head(3)

Zbiór treningowy: 6815 zwrotek
Zbiór testowy:    1203 zwrotek
Liczba unikalnych etykiet ze zbioru treningowego: 20


,song,section,text,label
0,Chcę żyć,zwrotka,Zawsze byłem i będę twardy\nJak każdy myśli o ...,Sobel
1,Kapie,zwrotka,Śmieszy mnie to co nazywacie trapem\nKilka but...,Kabe
2,Frank Lentini,intro,- Kurwa już trzeci raz ci się nagrywam na tą p...,Diho


## Kodowanie etykiet


In [3]:
labels = sorted(train_df["label"].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
NUM_LABELS = len(labels)

# Usunięcie ze zbioru testowego zwrotek o nieznanych etykietach (niewystępujących w zbiorze treningowym)
test_df = test_df[test_df["label"].isin(label2id)].reset_index(drop=True)

# Mapowanie etykiet na indeksy
train_df = train_df.copy()
train_df["labels"] = train_df["label"].map(label2id)
test_df["labels"] = test_df["label"].map(label2id)

# Wydzielenie zbioru walidacyjnego ze zbioru testowego i pomniejszenie zbioru testowego
val_df, test_df = train_test_split(
    test_df,
    test_size=0.5,
    random_state=3407,
    stratify=test_df["labels"],
)

print(f"Liczba klas (artystów): {NUM_LABELS}")
print(f"Trening: {len(train_df)}, Walidacja: {len(val_df)}, Testowanie: {len(test_df)}")

Liczba klas (artystów): 20
Trening: 6815, Walidacja: 601, Testowanie: 602


## Wczytanie Gemma 4 E2B przez Unsloth


In [4]:
MODEL_NAME = "unsloth/gemma-4-E2B"
MAX_LENGTH = 600  # ok. 99. percentyl

# Konfiguracja kwantyzacji 4-bitowej przez BitsAndBytes
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Użycie kwantyzacji NF4 (Normal Float 4)
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME, # Nazwa modelu do pobrania z HuggingFace
    max_seq_length=MAX_LENGTH, # Maksymalna ilość tokenów na wejściu dla modelu
    load_in_4bit=False, # kwantyzacja przez BitsAndBytesConfig poniżej
    dtype=None,
    auto_model=AutoModelForSequenceClassification, # Model do klasyfikacji sekwencji (z głowicą klasyfikacyjną)
    num_labels=NUM_LABELS, # Liczba klas (artystów do klasyfikacji)
    id2label=id2label, # Mapowanie indeksów na etykiety
    label2id=label2id, # Mapowanie etykiet na indeksy
    trust_remote_code=False,
    ignore_mismatched_sizes=True,
    use_exact_model_name=True, # Wymuszenie użycia dokładnej nazwy modelu (bez automatycznego dopasowania do innej wersji)
    quantization_config=bnb_config, # Konfiguracja kwantyzacji 4-bitowej przez BitsAndBytes
    device_map={"": 0}, # Wymuszenie użycia GPU 0
)

# Konfiguracja tokenizera dla modelu Gemma 4 (szablon konwersacyjny)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Konfiguracja adapterów LoRA dla modelu Gemma 4 (tylko głowica klasyfikacyjna)
model = FastModel.get_peft_model(
    model,
    r=32, # Rząd macierzy LoRA
    lora_alpha=32,
    lora_dropout=0.05, # Prawdopodobieństwo wyzerowania wag LoRA podczas treningu (dropout)
    bias="none",
    task_type=TaskType.SEQ_CLS, # Typ zadania ustawiony na klasyfikację sekwencji
    modules_to_save=["score"], # Zapisanie tylko modułów LoRA w głowicy klasyfikacyjnej (score)
    finetune_vision_layers=False, # Wyłączenie trenowania warstw wizyjnych
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    use_gradient_checkpointing="unsloth", # Unsloth zarządza gradient checkpointingiem
    random_state=3407,
)

# Wymuszenie gradientów wejściowych dla adapterów LoRA
# Bez tego gradient checkpointing odcina gradienty od adapterów LoRA (głowica klasyfikacyjna widzi znikome gradienty i loss stoi w miejscu).
model.enable_input_require_grads() 
model.print_trainable_parameters()

==((====))==  Unsloth 2026.6.7: Fast Gemma4 patching. Transformers: 5.10.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Gemma4ForSequenceClassification LOAD REPORT from: unsloth/gemma-4-E2B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: Allowing gradients for `base_model.model.score` since it's in `modules_to_save`.
trainable params: 48,347,136 || all params: 5,152,675,360 || trainable%: 0.9383


## Formatowanie wejścia i tokenizacja datasetów


In [5]:
# Prompt do klasyfikacji (bo Gemma 4 jest konwersacyjna)
CLASSIFICATION_PROMPT = """Przeczytaj poniższą zwrotkę polskiego utworu rapowego i określ, kto jest wykonawcą.

Tekst zwrotki:
{verse}"""

# Funkcja formatująca zwrotkę do postaci promptu dla modelu konwersacyjnego
def format_input(verse: str) -> str:
    messages = [{"role": "user", "content": CLASSIFICATION_PROMPT.format(verse=verse)}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return text.removeprefix("<bos>")

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

# Przygotowanie datasetów HuggingFace z formatowaniem tekstu i tokenizacją
keep_cols = ["text", "labels"]
for df in (train_df, val_df, test_df):
    df["verse"] = df["text"]
    df["text"] = df["verse"].map(format_input)

# Konwersja DataFrame'ów na datasety HuggingFace i tokenizacja
train_ds = Dataset.from_pandas(train_df[keep_cols], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[keep_cols], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[keep_cols], preserve_index=False)

# Tokenizacja datasetów (z usunięciem kolumny "text", bo po tokenizacji nie jest już potrzebna)
train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
test_ds = test_ds.map(tokenize_batch, batched=True, remove_columns=["text"])

# Data collator do dynamicznego paddingu podczas treningu
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Przykład wejścia:\n", train_df.iloc[0]["text"][:400], "\n...")
print(train_ds)

Map:   0%|          | 0/6815 [00:00<?, ? examples/s]

Map:   0%|          | 0/601 [00:00<?, ? examples/s]

Map:   0%|          | 0/602 [00:00<?, ? examples/s]

Przykład wejścia:
 <|turn>user
Przeczytaj poniższą zwrotkę polskiego utworu rapowego i określ, kto jest wykonawcą.

Tekst zwrotki:
Zawsze byłem i będę twardy
Jak każdy myśli o byciu martwym
Jak czasem myślę ile byłem warty
To zawsze widzę ten gnój, wiesz
Zawsze żyłem to rzucałem karty
Szymon wygra, a może spadnie
Dzisiaj kochać nie umie każdy
A jednak żyję życiem we dwóch
Nie wiem jak iść do przodu
I chyba nie chcę  
...
Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 6815
})


## Definicja metryk, wagi klas i label smoothing


In [6]:
# Współczynnik label smoothing dla funkcji straty
LABEL_SMOOTHING = 0.1

# Obliczanie wag klas dla funkcji straty (klasy mniej liczne dostają większą wagę, co pomaga modelowi lepiej uczyć się z niezbalansowanych danych)
label_counts = train_df["labels"].value_counts()
class_weights = torch.tensor(
    [len(train_df) / (NUM_LABELS * label_counts[i]) for i in range(NUM_LABELS)],
    dtype=torch.float32,
)

# Funkcja obliczająca metryki do ewaluacji (accuracy, macro F1, top-3 accuracy)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    top3 = np.argsort(logits, axis=-1)[:, -3:]
    top3_acc = np.mean([labels[i] in top3[i] for i in range(len(labels))])
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "top3_accuracy": top3_acc,
    }

## Konfiguracja parametrów treningu


In [7]:
# Custom Trainer, który uwzględnia wagi klas i label smoothing w funkcji straty
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, label_smoothing=0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.label_smoothing = label_smoothing

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = torch.nn.functional.cross_entropy(
            logits,
            labels,
            weight=weight,
            label_smoothing=self.label_smoothing,
        )
        return (loss, outputs) if return_outputs else loss

use_cuda = torch.cuda.is_available()

# Konfiguracja treningu
training_args = TrainingArguments(
    output_dir="outputs-gemma4-rap-classifier-unsloth",
    num_train_epochs=6, # Liczba epok treningu
    per_device_train_batch_size=8, # Batch size podczas treningu
    per_device_eval_batch_size=16, # Batch size podczas ewaluacji
    gradient_accumulation_steps=2, # Akumulacja gradientów co 2 kroki
    learning_rate=2e-4, # Krok uczenia
    weight_decay=0.01, # Parametr regularyzacji
    warmup_ratio=0.1, # Procent kroków rozgrzewkowych
    lr_scheduler_type="linear", # Liniowy scheduler dla tempa uczenia
    optim="adamw_8bit", # Optymalizator
    eval_strategy="epoch", # Ewaluacja po każdej epoce
    save_strategy="epoch", # Zapisywanie modelu po każdej epoce
    load_best_model_at_end=True, # Ładowanie najlepszego modelu na koniec treningu
    metric_for_best_model="eval_f1_macro", # Metryka do wyboru najlepszego modelu
    greater_is_better=True, # W przypadku F1 większa wartość jest lepsza
    train_sampling_strategy="group_by_length",
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    tf32=use_cuda and torch.cuda.is_tf32_supported(),
    logging_steps=10,
    seed=3407,
    bf16=use_cuda and torch.cuda.is_bf16_supported(),
    fp16=use_cuda and not torch.cuda.is_bf16_supported(),
    report_to="none",
)

# Inicjalizacja customowego trenera z uwzględnieniem wag klas i label smoothing
trainer = WeightedTrainer(
    model=model, # Model do trenowania
    args=training_args, # Konfiguracja treningu
    train_dataset=train_ds, # Dataset treningowy
    eval_dataset=val_ds, # Dataset walidacyjny
    processing_class=tokenizer, # Przetwarzanie danych przez tokenizer
    data_collator=data_collator, # Dynamiczny padding
    compute_metrics=compute_metrics, # Funkcja obliczająca metryki do ewaluacji
    class_weights=class_weights, # Wagi klas dla funkcji straty
    label_smoothing=LABEL_SMOOTHING, # Współczynnik label smoothing dla funkcji straty
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)], # Wczesne zatrzymanie treningu, jeśli metryka ewaluacji nie poprawi się przez 2 kolejne epoki
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## Trening modelu


In [8]:
train_result = trainer.train()

print(f"\nCzas treningu: {train_result.metrics['train_runtime']:.0f} s")
print(f"Końcowy loss: {train_result.metrics['train_loss']:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,815 | Num Epochs = 6 | Total steps = 2,556
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 48,347,136 of 5,152,675,360 (0.94% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Top3 Accuracy
1,3.413991,3.370448,0.079867,0.040377,0.232945
2,3.365453,3.379019,0.096506,0.044235,0.266223
3,3.329883,3.157104,0.164725,0.106464,0.324459
4,2.984972,3.032411,0.221298,0.170106,0.387687
5,2.709893,2.860301,0.261231,0.245408,0.499168
6,2.652306,2.775052,0.342762,0.311872,0.574043


Unsloth: Restored added_tokens_decoder metadata in outputs-gemma4-rap-classifier-unsloth/checkpoint-426/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs-gemma4-rap-classifier-unsloth/checkpoint-852/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs-gemma4-rap-classifier-unsloth/checkpoint-1278/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs-gemma4-rap-classifier-unsloth/checkpoint-1704/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs-gemma4-rap-classifier-unsloth/checkpoint-2130/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs-gemma4-rap-classifier-unsloth/checkpoint-2556/tokenizer_config.json.



Czas treningu: 3978 s
Końcowy loss: 3.1094


## Ewaluacja na zbiorze testowym


In [9]:
test_metrics = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix="test")
print(f"Accuracy:      {test_metrics['test_accuracy']:.2%}")
print(f"Macro-F1:      {test_metrics['test_f1_macro']:.4f}")
print(f"Top-3 accuracy: {test_metrics['test_top3_accuracy']:.2%}")

early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,Top3 Accuracy
2.652306,2.788230,6,0.350498,0.335167,0.544850


Accuracy:      35.05%
Macro-F1:      0.3352
Top-3 accuracy: 54.49%


### Sprawdzenie dokładności w zależności od wykonawcy


In [10]:
predictions = trainer.predict(test_ds)
pred_ids = np.argmax(predictions.predictions, axis=-1)

results_df = pd.DataFrame({
    "label": test_df["label"],
    "prediction": [id2label[i] for i in pred_ids],
})
results_df["correct"] = results_df["label"] == results_df["prediction"]

print("Dokładność w zależności od wykonawcy:")
per_label = (
    results_df.groupby("label")["correct"]
    .agg(["mean", "count"])
    .sort_values("count", ascending=False)
)
print(per_label.to_string())

Dokładność w zależności od wykonawcy:
                   mean  count
label                         
Sentino        0.030769     65
Young Igi      0.090909     55
White 2115     0.068182     44
Quebonafide    0.000000     44
Zeamsone       0.068182     44
Malik Montana  0.023810     42
Pezet          0.048780     41
Bedoes         0.075000     40
Żabson         0.028571     35
Sobel          0.058824     34
Mata           0.000000     32
OKI            0.037037     27
Kabe           0.043478     23
Diho           0.045455     22
Guzior         0.000000     16
Avi            0.153846     13
Young Leosia   0.000000     11
Oki            0.000000      8
OG Olgierd     0.000000      4
Louis Villain  0.000000      2


## Predykcja dla pojedynczych zwrotek


In [11]:
@torch.no_grad()
def predict_artist(verse: str, top_k: int = 3):
    model.for_inference()
    inputs = tokenizer(
        format_input(verse),
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(model.device)
    probs = model(**inputs).logits.softmax(-1)[0]
    top = probs.topk(min(top_k, NUM_LABELS))
    return [(id2label[i.item()], p.item()) for p, i in zip(top.values, top.indices)]


samples = test_df.sample(5, random_state=7)
for _, row in samples.iterrows():
    ranking = predict_artist(row["verse"])
    top_label = ranking[0][0]
    status = "TRAFIONO" if top_label == row["label"] else "POMYŁKA"
    print(f"[{status}] utwór: {row['song']}")
    print(f"  Fragment:   {' / '.join(row['verse'].splitlines()[:2])}")
    print(f"  Prawda:     {row['label']}")
    print(f"  Predykcja:  " + ", ".join(f"{a} ({p:.0%})" for a, p in ranking) + "\n")

[POMYŁKA] utwór: Plan B
  Fragment:   Ty miałaś wszystkie noce dla mnie, gdy były wszystkie oczy na mnie wtedy / A jak byłem na dnie, to szukałem głębi
  Prawda:     Pezet
  Predykcja:  Avi (50%), White 2115 (8%), Diho (7%)

[TRAFIONO] utwór: Normalnie żyjąc
  Fragment:   Co to w ogóle oznacza gdy mi mówisz, że inni normalnie żyją / Gdy reszta haftuje na melanżach moi ludzie linie kozackie szyją
  Prawda:     Guzior
  Predykcja:  Guzior (41%), Bedoes (12%), Avi (6%)

[POMYŁKA] utwór: Kołysanka
  Fragment:   Stawiam na zero, stawiam na zero, ale jakoś wychodzę na plus / Nie wierzę kobietom, nie wierzę uśmiechom
  Prawda:     Malik Montana
  Predykcja:  Young Igi (28%), White 2115 (13%), Louis Villain (10%)

[TRAFIONO] utwór: bby blue
  Fragment:   (Yeah) Yeah (Yeah), yeah, chodź, pomaluję Twój świat (Świat) / Pomaluję na niebiesko (Świat), tylko, kurwa, na niebiesko
  Prawda:     OKI
  Predykcja:  OKI (60%), Louis Villain (7%), Żabson (6%)

[POMYŁKA] utwór: R.U.N.
  Fragment:   Te wszys

In [12]:
custom_verse = """\
Pamiętam dobrze szare bloki, beton i te klatki
Z chłopakami na osiedlu wieczne sprzeczki, przepychanki
Dziś na scenie cały kraj mnie zna, choć droga była śliska
Z mikrofonem w ręce mówię prawdę prosto z blokowiska"""

for artist, prob in predict_artist(custom_verse):
    print(f"{artist:30s} {prob:.1%}")

Bedoes                         45.0%
Kabe                           16.3%
Malik Montana                  6.8%


## Zapis adapterów LoRA i głowicy klasyfikacyjnej


In [13]:
SAVE_DIR = "gemma4-e2b-polish-rap-classifier"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Adaptery LoRA i klasyfikator zapisane w: {SAVE_DIR}")

Unsloth: Restored added_tokens_decoder metadata in gemma4-e2b-polish-rap-classifier/tokenizer_config.json.


Adaptery LoRA i klasyfikator zapisane w: gemma4-e2b-polish-rap-classifier
